# 10a — Cost-benefit analysis

Opens Chapter 10 ("Transition Risk"), which — unlike Chapter 9's
carbon-accounting focus — applies welfare/public economics to climate
transition policy. This notebook covers `chap10_cba1.m` and `chap10_
cba2.m`: welfare-weighted cost-benefit analysis, and standard project-
appraisal metrics (NPV/BCR/PI/IRR). Both scripts are fully
self-contained (no external data).

`internal_rate_return.m` ships as a plain function file directly in
this chapter's own archive folder (not the shared `hfs-archive`
toolbox), so its logic is fully visible and ported faithfully below:
find the roots of the cashflow polynomial, convert each to a rate via
$1/\text{root} - 1$, keep only real roots, and return the largest (or
`NaN` if none exist).

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import brentq

## 1. Welfare-weighted cost-benefit analysis (`chap10_cba1.m`)

Two policy-affected groups with incomes $Y_1=1000$, $Y_2=500$ and
income changes $dY_1$, $dY_2$ under two scenarios. Three welfare
aggregation rules are compared: a **utilitarian** rule weighting each
group's marginal utility of income ($1/2\sqrt{Y}$, i.e. diminishing
marginal utility under $\sqrt{\cdot}$ utility), a **Rawlsian** rule
that only counts the worse-off group (`omega1=0`), and a simple
**Kaldor-Hicks** rule that just sums raw income changes with no
distributional weighting at all. The same comparison is then redone
using the exact (non-linearized) utility change
$\sqrt{Y+dY}-\sqrt{Y}$ rather than the linear/marginal approximation.

In [2]:
Y1, Y2 = 1000, 500
dY1 = np.array([50, 40])
dY2 = np.array([-25, -30])

# Marginal (linearized) welfare change
omega1, omega2 = 1, 1
dW_U = omega1 * (1 / (2 * np.sqrt(Y1))) * dY1 + omega2 * (1 / (2 * np.sqrt(Y2))) * dY2
omega1, omega2 = 0, 1
dW_Rawls = omega1 * (1 / (2 * np.sqrt(Y1))) * dY1 + omega2 * (1 / (2 * np.sqrt(Y2))) * dY2
dW_KH = dY1 + dY2

dW = pd.DataFrame({"Utilitarian": dW_U, "Rawlsian": dW_Rawls, "Kaldor-Hicks": dW_KH},
                   index=["Scenario 1", "Scenario 2"])
print("Marginal (linearized) welfare change:")
display(dW.round(3))

# Exact (non-linearized) welfare change
omega1, omega2 = 1, 1
Delta_W_U = omega1 * (np.sqrt(Y1 + dY1) - np.sqrt(Y1)) + omega2 * (np.sqrt(Y2 + dY2) - np.sqrt(Y2))
omega1, omega2 = 0, 1
Delta_W_Rawls = omega1 * (np.sqrt(Y1 + dY1) - np.sqrt(Y1)) + omega2 * (np.sqrt(Y2 + dY2) - np.sqrt(Y2))
Delta_W_KH = dY1 + dY2

Delta_W = pd.DataFrame({"Utilitarian": Delta_W_U, "Rawlsian": Delta_W_Rawls, "Kaldor-Hicks": Delta_W_KH},
                        index=["Scenario 1", "Scenario 2"])
print("\nExact welfare change:")
Delta_W.round(3)

Marginal (linearized) welfare change:


,Utilitarian,Rawlsian,Kaldor-Hicks
Scenario 1,0.232,-0.559,25
Scenario 2,-0.038,-0.671,10



Exact welfare change:


,Utilitarian,Rawlsian,Kaldor-Hicks
Scenario 1,0.215,-0.566,25
Scenario 2,-0.055,-0.681,10


## 2. Project appraisal: NPV, BCR, PI, IRR (`chap10_cba2.m`)

Two illustrative cost/benefit cashflow schedules over 9 periods, each
evaluated with the standard project-appraisal toolkit: Net Present
Value, Benefit-Cost Ratio, Profitability Index (at a 3% discount rate),
and the Internal Rate of Return (via `internal_rate_return`, the
chapter's own polynomial-roots IRR solver). The final section finds the
discount rate $\varrho^{\star}$ at which the two projects' NPVs are
equal, via bisection.

In [3]:
def internal_rate_return(cf):
    # Faithful port of chap10's internal_rate_return.m: treat the cashflow
    # as polynomial coefficients (most recent period first, per MATLAB's
    # `rev`), find its roots, convert each root r to a rate via 1/r - 1,
    # keep only real roots, and return the largest (or NaN if none exist).
    p = cf[::-1]
    roots = np.roots(p)
    rates = 1 / roots - 1
    real_rates = rates[np.abs(rates.imag) < 1e-9].real
    if len(real_rates) == 0:
        return np.nan
    return real_rates.max()

varrho = np.array([0.00, 0.01, 0.03, 0.05, 0.07])
t = np.arange(9)

def appraise(Ct, Bt, label):
    C0 = Ct[0]
    CF = Bt - Ct
    irr = internal_rate_return(CF)
    PV_C = np.sum(Ct / (1 + varrho[2]) ** t)  # varrho = 3% for the summary metrics
    PV_B = np.sum(Bt / (1 + varrho[2]) ** t)
    NPV = PV_B - PV_C
    BCR = PV_B / PV_C
    PI = NPV / C0
    print(f"--- {label} (at varrho=3%) ---")
    print(f"NPV = {NPV:.2f}, BCR = {BCR:.2f}, PI = {PI:.2f}, IRR = {100*irr:.2f}%")
    return CF

Ct1 = np.array([100, 25, 25, 0, 0, 0, 0, 0, 0], dtype=float)
Bt1 = np.array([0, 10, 20, 30, 50, 50, 50, 50, 50], dtype=float)
CF1 = appraise(Ct1, Bt1, "Project 1")

Ct2 = np.array([80, 15, 15, 0, 0, 0, 0, 0, 0], dtype=float)
Bt2 = np.array([0, 10, 20, 30, 40, 40, 40, 40, 40], dtype=float)
CF2 = appraise(Ct2, Bt2, "Project 2")

def Delta_NPV(rho):
    return np.sum((CF1 - CF2) / (1 + rho) ** t)

varrho_star = brentq(Delta_NPV, 0, 0.10)
print(f"\nBreak-even discount rate varrho* = {100*varrho_star:.2f}%  "
      f"(the discount rate at which Project 1 and Project 2 have equal NPV)")

--- Project 1 (at varrho=3%) ---
NPV = 117.73, BCR = 1.80, PI = 1.18, IRR = 17.42%
--- Project 2 (at varrho=3%) ---
NPV = 114.96, BCR = 2.06, PI = 1.44, IRR = 21.66%

Break-even discount rate varrho* = 4.37%  (the discount rate at which Project 1 and Project 2 have equal NPV)
